# Week 4, Day 4 — Deep Agents, the Top Layer
### Local Models Edition — by Abhishek

A Deep Agent wraps yesterday's `create_agent` in an opinionated harness for
serious, multi-step work: a **planning tool**, a **filesystem**, and
**sub-agents**. You supply the intent; the harness supplies the structure.

**Local/free throughout:** search via DuckDuckGo (no key), slides via
`python-pptx` (no key) instead of a hosted design API.

## When to reach for this
For a quick question with one or two tools, `create_agent` is the right tool
and a Deep Agent is overkill. Deep Agents earn their keep when a task has
many steps, produces artifacts, and benefits from the agent organizing its
own work — research and report writing is the classic example, so that's
what we build.


This lab writes files into a `sandbox/` folder next to this notebook. Run
this notebook from inside `4_langchain_langgraph/` so the paths line up.


## 0. Setup

In [ ]:
import importlib.util
IN_COLAB = importlib.util.find_spec("google.colab") is not None
BACKEND = "huggingface" if IN_COLAB else "ollama"
print(f"Backend: {BACKEND}")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q deepagents langchain-huggingface transformers torch accelerate duckduckgo-search python-pptx
else:
    %pip install -q deepagents langchain-ollama ollama duckduckgo-search python-pptx


In [ ]:
import os
from langchain_core.tools import tool
from duckduckgo_search import DDGS
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

if BACKEND == "huggingface":
    from transformers import pipeline
    from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
    pipe = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", max_new_tokens=300)
    model = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=pipe))
else:
    from langchain_ollama import ChatOllama
    model = ChatOllama(model="llama3.2:3b", temperature=0.3)

print("model ready")


## A web search tool

In [ ]:
@tool
def search(query: str) -> str:
    """Search the web and return a short summary of the top results."""
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=3))
    if not results:
        return "No results found."
    return "\n\n".join(f"{r['title']}: {r['body']}" for r in results)


## A research agent that plans, searches and writes

**Scenario:** a company is getting ready to move its sales fleet to electric
vehicles. Someone has to do the homework first: how ready is the public
charging network, and which providers could a fleet actually rely on?

We point the agent's filesystem at a real `sandbox` folder, so anything it
writes appears on disk.


In [ ]:
sandbox = os.path.abspath("sandbox")
os.makedirs(sandbox, exist_ok=True)

researcher = create_deep_agent(
    model=model,
    tools=[search],
    system_prompt=(
        "You are a research analyst. Plan your work with your todo tool, "
        "research with the search tool, and write your findings as a tidy markdown briefing to a file."
    ),
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True),
)


In [ ]:
brief = """
Our company is planning to move its sales fleet to electric vehicles.
Research the public EV charging landscape in the US: find out roughly how many public charging points there are,
and pick out two major charging networks a fleet could rely on.
Write a one page markdown briefing, with a heading and a short section for each, to the file charging.md.
"""

result = researcher.invoke({"messages": [{"role": "user", "content": brief}]})
print(result["messages"][-1].content)


In [ ]:
tools_used = [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", []) or [])]
print("Tools the agent called, in order:")
print(tools_used)


## Sub-agents: handing off focused work

A sub-agent is a helper the main agent delegates to through its `task` tool.
It runs with its own fresh context, does one job, and reports back a tidy
result. Local models especially benefit from this — a smaller model juggling
one focused sub-task tends to do better than the same model juggling
everything at once.


In [ ]:
research_ev_instructions = """
You research one electric vehicle using the search tool and return three concise facts
that a fleet buyer would care about, such as price, range and charging.
"""

overall_instructions = """
You write comparison briefings for a company choosing electric vehicles for its sales fleet.
For each vehicle, delegate the research to your vehicle-researcher sub-agent,
then write a markdown comparison to a file, ending with a clear recommendation.
"""

research_subagent = {
    "name": "vehicle-researcher",
    "description": "Researches a single electric vehicle and returns a short list of facts about it.",
    "system_prompt": research_ev_instructions,
}

lead = create_deep_agent(
    model=model,
    tools=[search],
    system_prompt=overall_instructions,
    subagents=[research_subagent],
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True),
)


In [ ]:
mission = """
Compare the Tesla Model Y and the Ford Mustang Mach-E as candidates for our 100-car sales fleet.
Research each vehicle, then write a short markdown comparison with a recommendation to fleet.md.
"""

result = lead.invoke({"messages": [{"role": "user", "content": mission}]})
tools_used = [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", []) or [])]
print("Tools the lead agent called:", tools_used)


### Now go and look at `fleet.md` in the `sandbox` folder

## One more thing: Agent Skills

`deepagents` supports Anthropic's Agent Skills pattern — a `SKILL.md` file
with YAML frontmatter (name + description) followed by instructions. Only the
name/description go into the system prompt; the agent reads the full file
with its own `read_file` tool when it decides the skill is relevant
(progressive disclosure). We've provided one at
[`sandbox/skills/fleet-slide/SKILL.md`](sandbox/skills/fleet-slide/SKILL.md)
— open it and have a read before handing it to an agent.


## A sub-agent that makes the slide

We define a slide-maker sub-agent with the fleet-slide skill and a
`create_slide` tool of its own — backed by `slide_kit.py`, which uses
`python-pptx` to build a branded PowerPoint slide entirely locally, no design
API involved.


In [ ]:
from slide_kit import build_slide

@tool
def create_slide(title: str, key_points: list[str], recommendation: str) -> str:
    """Create a one-slide PowerPoint in the house style, saved as fleet.pptx."""
    build_slide(title, key_points, recommendation, os.path.join(sandbox, "fleet.pptx"))
    return "Saved the slide to /fleet.pptx"

slide_maker = {
    "name": "slide-maker",
    "description": "Turns a finished recommendation into a one-slide PowerPoint deck.",
    "system_prompt": "You turn research recommendations into slides, following your fleet-slide skill.",
    "tools": [create_slide],
    "skills": ["/skills/"],
}

presenter = create_deep_agent(
    model=model,
    subagents=[slide_maker],
    system_prompt="You prepare research for presentation by delegating to your slide-maker sub-agent.",
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True),
)


In [ ]:
result = presenter.invoke({"messages": [{"role": "user", "content":
    "Read fleet.md and have a one-slide deck made of its recommendation."}]})
print(result["messages"][-1].content)


The sandbox now contains `fleet.pptx` — open it and take a look at a slide
that a local, free model researched, wrote, and designed end to end.

## Recap, and where we are heading
You ran a Deep Agent that planned its own work, searched the web, wrote files
to disk, delegated to a sub-agent, and turned its findings into a branded
PowerPoint slide by following an Agent Skill — all on a local model.

Tomorrow's Sidekick deliberately drops back to `create_agent` for a
responsive, interactive assistant.

## Exercise
Give the research agent a second tool — the Playwright browser tools from
Day 3 work here too, remember they're async, so switch `.invoke(...)` to
`await .ainvoke(...)`. Then read every file the agent wrote for itself.
**Stretch:** write your own `SKILL.md` teaching the slide-maker your own
house style instead of the example brand.
